In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)

In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df_delivery.shape}")
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df_delivery.copy().drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_clean.isnull().sum())

df_clean = df_delivery.dropna()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

df_clean = df_clean.drop_duplicates()
check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder
categories = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
#print('data before encoding:\n', df_delivery[categories]) #show before encoding

le = LabelEncoder()
for cat in categories:
  df_clean[cat] = le.fit_transform(df_clean[cat])

#onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
#for cat in categories:
#  df_delivery[cat] = onehot_encoder.fit_transform(df_delivery[cat])
#data_onehot_encoded = onehot_encoder.fit_transform(df_delivery[categories]) # Apply fit_transform to the copied

#print('\nData after encoding:\n', data_onehot_encoded) #show after encoding
#for cat in categories:
 # df_delivery[cat] = data_onehot_encoded[cat]

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

# This plot shows that the delivery time is balanced

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import StratifiedKFold, KFold
X = df_clean.drop('Delivery_Time', axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.metrics import mean_absolute_error
n_splits = 5  # K=5 Folds
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)

model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)

mae_scores = []
for fold_idx, (train_index, test_index) in enumerate(kfold.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train and predict
  model.fit(X_train, y_train)
  y_fold_pred = model.predict(X_test)

  # Calculate metrics
  mae_scores.append(mean_absolute_error(y_test, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.plot(mae_scores, label='MAE')
plt.xlabel('Iteration')
plt.ylabel('MAE')
plt.title('Linear Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task Bonus: Write your code here: